# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets, fields and field IDs
def get_record_sets(md):
    # Try direct access, otherwise walk document
    if hasattr(md, "record_sets"):
        return md.record_sets
    if hasattr(md, "record_set"):
        return md.record_set
    # Fallback for croissant <1.0 or different key
    if hasattr(md, "recordSets"):
        return md.recordSets
    if hasattr(md, "recordSet"):
        return md.recordSet
    return []

record_sets = get_record_sets(metadata)
if not record_sets or len(record_sets) == 0:
    # Try to infer from ML Croissant Dataset expansion
    print("No record sets found in metadata. Attempting to enumerate record sets from dataset schema...")
    try:
        # mlcroissant uses dataset._metadata._expanded_jsonld to get the underlying JSON-LD
        expanded = dataset._metadata._expanded_jsonld
        # Find all entities of type 'cr:RecordSet'
        recs = []
        for entity in expanded:
            types = entity.get('@type', [])
            if isinstance(types, str):
                types = [types]
            if any(["http://mlcommons.org/croissant/RecordSet" in t or t.endswith("RecordSet") for t in types]):
                recs.append(entity)
        record_sets = recs
    except Exception as ex:
        print("Failed to parse record sets from expanded schema.", ex)

if not record_sets or len(record_sets) == 0:
    raise RuntimeError("No record sets found in dataset.")

print(f"Found {len(record_sets)} record set(s):")
record_set_ids = []
for idx, rec in enumerate(record_sets):
    if isinstance(rec, dict):
        rec_id = rec.get('@id', None)
        rec_name = rec.get('name', None) or rec.get('schema:name', None)
        rec_fields = rec.get('field', None) or rec.get('cr:field', None) or rec.get('fields', None)
    else:
        rec_id = getattr(rec, '@id', None)
        rec_name = getattr(rec, 'name', getattr(rec, 'schema:name', None))
        rec_fields = getattr(rec, 'field', getattr(rec, 'cr:field', None))
    print(f"  [{idx}]: @id={rec_id}, name={rec_name}")
    record_set_ids.append(rec_id)
    # Print field-level details for each record set
    field_items = rec_fields
    if field_items is not None:
        if isinstance(field_items, dict):
            field_items = [field_items]
        print("    Fields:")
        for f in field_items:
            if isinstance(f, dict):
                field_id = f.get('@id', None)
                field_name = f.get('name', None) or f.get('schema:name', None)
            else:
                field_id = getattr(f, '@id', None)
                field_name = getattr(f, 'name', getattr(f, 'schema:name', None))
            print(f"     - @id={field_id}, name={field_name}")
    else:
        # If field definitions not present here, skip
        pass

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# Use the list of record set ids found above
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0 and isinstance(records[0], dict):
            df = pd.DataFrame(records)
        else:
            # possibly single column
            df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set @id={record_set_id} with shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as ex:
        print(f"Failed to extract record set {record_set_id}: {ex}")

# For demonstration, use the first record set found
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"First few records of record set '@id={first_rs}':")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis from your DataFrame, using the field's @id
# You may need to adjust the @id to match what you found in the overview code above.
rs_id = record_set_ids[0]
df = dataframes[rs_id]

print(f"Columns in chosen record set '@id={rs_id}': {df.columns.tolist()}")

# Heuristically pick a numeric field: look for a column with number in the name, such as 'coverage', 'count', or similar
import numpy as np
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or df[col].dtype==float]
if not possible_numeric_fields:
    # Also check for columns that can be cast to numeric
    num_cols = []
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna().sample(min(5, len(df))), errors='raise')
            num_cols.append(col)
        except Exception:
            continue
    possible_numeric_fields = num_cols
print(f"Possible numeric fields: {possible_numeric_fields}")

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    raise RuntimeError("No numeric field found in the first record set. Please inspect columns and choose a numeric field for EDA.")

# Try to use a representative threshold
if df[numeric_field].dtype not in [np.float64, np.int64]:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = np.nanmean(df[numeric_field]) if np.nanmean(df[numeric_field]) and not np.isnan(np.nanmean(df[numeric_field])) else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (using field @id: {numeric_field}):")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records (z-score normalization):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (heuristically pick one that's not numeric)
cat_fields = [col for col in df.columns if col not in possible_numeric_fields and df[col].nunique() < 20]

group_field = cat_fields[0] if cat_fields else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by '{group_field}' (field @id: {group_field}):")
    display(grouped_df.head())
else:
    print("No categorical field found for grouping. Skipping group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Distribution of the normalized field
plt.figure(figsize=(7, 4))
plt.hist(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=20, color="skyblue", edgecolor="k")
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel("Frequency")
plt.title(f"Distribution of Normalized '{numeric_field}'")
plt.show()

# If group field exists, boxplot
if group_field is not None:
    plt.figure(figsize=(9,5))
    filtered_df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"'{numeric_field}' by group '{group_field}'")
    plt.suptitle("")
    plt.xlabel(f"{group_field}")
    plt.ylabel(f"{numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The `mlcroissant` library enables simplified loading and analysis of FAIR-compliant datasets described in Croissant schema, using `@id` fields for robust data referencing.
* We explored the protein mass spectrometry dataset, listing its record sets and sample fields. One numeric field was filtered, normalized, and visualized.
* These patterns of programmatic schema browsing and robust field selection enable reproducible science and can be generalized for other Croissant-format datasets.